<a href="https://colab.research.google.com/github/Lusic12/Run_parseq_ocr/blob/main/demo_parseg_VietNamese.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
! unzip '/content/drive/MyDrive/weights_ocr.zip' -d '/content/checkpoint'

Archive:  /content/drive/MyDrive/weights_ocr.zip
   creating: /content/checkpoint/weights/
   creating: /content/checkpoint/weights/rec/
  inflating: /content/checkpoint/weights/rec/VietOCR-best.pth  
  inflating: /content/checkpoint/weights/rec/best-parseq.ckpt  
   creating: /content/checkpoint/weights/detect/
  inflating: /content/checkpoint/weights/detect/craft_mlt_25k.pth  
  inflating: /content/checkpoint/weights/detect/model_0033999.pth  
  inflating: /content/checkpoint/weights/detect/model_yolov8.pt  


In [ ]:
!git clone https://github.com/Lusic12/Run_parseq_ocr.git

Cloning into 'Run_parseq_ocr'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 68 (delta 13), reused 63 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 24.31 KiB | 12.15 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [ ]:
%cd /content/Run_parseq_ocr/

/content/Run_parseq_ocr


In [ ]:
!pip install pip==23.0.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 27.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 14.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 54.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 582.1/582.1 kB 44.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.6/750.6 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.2/869.2 kB 59.1 MB/s eta 0:00:00
  Attempting uninstall: torch
    Found existing installation: torch 2.4.1+cu121
    Uninstalling torch-2.4.1+cu121:
      Successfully uninstalled torch-2.4.1+cu121
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.19.1+cu121
    Uninstalling torchvision-0.19.1+cu121:
      Successfully uninstalled torchvision-0.19.1+cu121
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the followin

In [ ]:
%cd /content/Run_parseq_ocr

/content/Run_parseq_ocr


In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
from PIL import Image

# Function to crop an image in memory based on bounding box points
def crop_image_in_memory(image, bd_pts):
    """Crop image in memory based on the bounding box."""
    left = bd_pts[0][0]
    top = bd_pts[0][1]
    right = bd_pts[2][0]
    bottom = bd_pts[2][1]
    cropped_image = image.crop((left, top, right, bottom))
    return cropped_image

# Process individual frames to extract cropped regions based on bounding boxes
def process_frames(predictor, frame_data, image_dir):
    images = []
    frame_id = frame_data['frame_id']
    image_path = os.path.join(image_dir, frame_id)

    if os.path.exists(image_path):
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f"Error opening image {image_path}: {e}")
            return []

        detection_results = frame_data.get("detection_results", [])

        # Crop each bounding box in the frame
        for result in detection_results:
            bd_pts = result.get("bd_pts", [])
            if len(bd_pts) != 4:
                print(f"Invalid bounding box for frame {frame_id}: {bd_pts}")
                continue
            try:
                cropped_image = crop_image_in_memory(image, bd_pts)
                images.append(cropped_image)
            except Exception as e:
                print(f"Error cropping image {frame_id}: {e}")
                continue

    return images

# Process a batch of frames concurrently
def process_batch(predictor, batch_data, image_dir):
    batch_images = []
    for frame_data in batch_data:
        images = process_frames(predictor, frame_data, image_dir)
        batch_images.extend(images)
    return batch_images

# Main function to process images from JSON data and apply prediction
def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, batch_size=16):
    """
        Process images by reading bounding box data from a JSON file.
        Predict character from cropped images.
        Args:
            json_file (str): Path to the JSON file containing bounding box information.
            image_dir (str): Directory containing the images.
            checkpoint_path (str): Path to the checkpoint file for the PARSeq model.
    """
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Initialize predictor with checkpoint path
    predictor = PARSeqPredictor(checkpoint_path)

    # Extract all frames and bounding box information from the JSON file
    frame_ids = list(data.keys())
    frame_data_list = [{'frame_id': frame_id, 'detection_results': data[frame_id].get('detection_results', [])} for frame_id in frame_ids]

    results = []

    # Use ThreadPoolExecutor to process batches in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for i in tqdm(range(0, len(frame_data_list), batch_size), desc="Processing Batches"):
            batch_data = frame_data_list[i:i + batch_size]
            futures.append(executor.submit(process_batch, predictor, batch_data, image_dir))

        # Wait for all futures to complete and gather results
        for future in as_completed(futures):
            try:
                batch_images = future.result()
                if batch_images:
                    # Perform predictions on cropped images
                    pred_texts, confidences = predictor.predict(batch_images)
                    results.extend(zip(pred_texts, confidences))
            except Exception as e:
                print(f"Error in thread: {e}")

    # Output results
    for pred_text, confidence in results:
        print(f"Predicted Text: {pred_text}, Confidence: {confidence:.4f}")

# Execute the program
if __name__ == "__main__":
    json_file = 'path/to/file.json'
    image_dir = 'path/to/image'
    checkpoint_path = 'path/to/checkpoint'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, batch_size=16)
    end_time = time.time()

    print(f"Total processing time: {end_time - start_time:.2f} seconds")


Processing Batches: 100%|██████████| 14/14 [00:00<00:00, 2833.58it/s]


Streaming output truncated to the last 5000 lines.
Predicted Text: ['AGRIBANK'], Confidence: 0.9998
Predicted Text: ['NGHÈO'], Confidence: 0.9957
Predicted Text: ['giây'], Confidence: 0.9394
Predicted Text: ['BÊNH'], Confidence: 0.9060
Predicted Text: ['NHÂN'], Confidence: 0.9998
Predicted Text: ['CHÍNH'], Confidence: 0.9999
Predicted Text: ['AGRIBANK'], Confidence: 0.9999
Predicted Text: ['RIBAN'], Confidence: 0.9475
Predicted Text: ['SU'], Confidence: 0.9729
Predicted Text: ['06:30:27'], Confidence: 0.9994
Predicted Text: ['HD'], Confidence: 0.9999
Predicted Text: ['HÀNG:'], Confidence: 0.9902
Predicted Text: ['JAGRIBANK'], Confidence: 0.9981
Predicted Text: ['tuc'], Confidence: 0.9222
Predicted Text: ['Dừng'], Confidence: 0.9968
Predicted Text: ['Xuyên'], Confidence: 0.9994
Predicted Text: ['với'], Confidence: 0.9990
Predicted Text: ['hải'], Confidence: 0.9988
Predicted Text: ['Tổng'], Confidence: 0.9985
Predicted Text: ['Viêt'], Confidence: 0.9945
Predicted Text: ['Hải'], Confidenc

In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import statistics
from PIL import Image
import torch
import sys

sys.path.append('/content/Run_parseq_ocr/parseq')
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint

class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
        img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
        return parseq, img_transform

    @torch.inference_mode()
    def predict(self, images):
        pred_texts = []
        confidences = []
        for image in images:
            pred_text, confidence = self.predict_parseq(image)
            pred_texts.append(pred_text)
            confidences.append(confidence)
        return pred_texts, confidences

    @torch.inference_mode()
    def predict_parseq(self, image):
        image = self.img_transform(image).unsqueeze(0).to(self.device)
        p = self.parseq(image).softmax(-1)
        pred, p = self.parseq.tokenizer.decode(p)
        return (pred, statistics.mean(p[0].tolist()))

def crop_image_in_memory(image, bd_pts):
    """Crop image in memory based on the bounding box."""
    left = bd_pts[0][0]
    top = bd_pts[0][1]
    right = bd_pts[2][0]
    bottom = bd_pts[2][1]
    cropped_image = image.crop((left, top, right, bottom))
    return cropped_image

def process_frames(predictor, frame_data, image_dir):
    images = []
    frame_id = frame_data['frame_id']
    image_path = os.path.join(image_dir, frame_id)

    if os.path.exists(image_path):
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f"Error opening image {image_path}: {e}")
            return []

        detection_results = frame_data.get("detection_results", [])

        # Crop all bounding boxes in the frame
        for result in detection_results:
            bd_pts = result.get("bd_pts", [])
            if len(bd_pts) != 4:
                print(f"Invalid bounding box for frame {frame_id}: {bd_pts}")
                continue
            try:
                cropped_image = crop_image_in_memory(image, bd_pts)
                images.append(cropped_image)
            except Exception as e:
                print(f"Error cropping image {frame_id}: {e}")
                continue

    return images

def process_batch(predictor, batch_data):
    batch_images = []
    for frame_data in batch_data:
        images = process_frames(predictor, frame_data, image_dir)
        batch_images.extend(images)
    return batch_images

def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, batch_size=16):
    """Process images by reading bounding box data from a JSON file."""
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Initialize the predictor with the checkpoint
    predictor = PARSeqPredictor(checkpoint_path)

    # Extract all frames from the JSON file
    frame_ids = list(data.keys())
    frame_data_list = [{'frame_id': frame_id, 'detection_results': data[frame_id].get('detection_results', [])} for frame_id in frame_ids]

    results = []

    # Use ThreadPoolExecutor to process multiple batches concurrently
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for i in tqdm(range(0, len(frame_data_list), batch_size), desc="Processing Batches"):
            batch_data = frame_data_list[i:i + batch_size]
            futures.append(executor.submit(process_batch, predictor, batch_data))

        # Wait for all futures to complete and handle results
        for future in as_completed(futures):
            try:
                batch_images = future.result()
                if batch_images:
                    # Predict for cropped images
                    pred_texts, confidences = predictor.predict(batch_images)
                    results.extend(zip(pred_texts, confidences))
            except Exception as e:
                print(f"Error in thread: {e}")

    # Print results
    for pred_text, confidence in results:
        print(f"Predicted Text: {pred_text}, Confidence: {confidence:.4f}")

# Execute the program
if __name__ == "__main__":
    json_file = '/content/drive/MyDrive/L07_new/results/L07_V002_keyframes_filtered.json'
    image_dir = '/content/drive/MyDrive/L07_new/L07_V002_keyframes_filtered'
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, batch_size=16)
    end_time = time.time()

    print(f"Tổng thời gian xử lý: {end_time - start_time:.2f} giây")


Processing Batches: 100%|██████████| 56/56 [00:00<00:00, 10414.62it/s]


KeyboardInterrupt: 

In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time
import statistics
from PIL import Image
import torch
import sys

sys.path.append('/content/Run_parseq_ocr/parseq')
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint

class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
        img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
        return parseq, img_transform

    @torch.inference_mode()
    def predict(self, images):
        pred_texts = []
        confidences = []
        for image in images:
            pred_text, confidence = self.predict_parseq(image)
            pred_texts.append(pred_text)
            confidences.append(confidence)
        return pred_texts, confidences

    @torch.inference_mode()
    def predict_parseq(self, image):
        image = self.img_transform(image).unsqueeze(0).to(self.device)
        p = self.parseq(image).softmax(-1)
        pred, p = self.parseq.tokenizer.decode(p)
        return (pred, statistics.mean(p[0].tolist()))

def crop_image_in_memory(image, bd_pts):
    """Crop image in memory based on the bounding box."""
    left = bd_pts[0][0]
    top = bd_pts[0][1]
    right = bd_pts[2][0]
    bottom = bd_pts[2][1]
    cropped_image = image.crop((left, top, right, bottom))
    return cropped_image

def process_frames(predictor, frame_data, image_dir):
    images = []
    frame_id = frame_data['frame_id']
    image_path = os.path.join(image_dir, frame_id)

    if os.path.exists(image_path):
        try:
            image = Image.open(image_path).convert('RGB')
            print(f"Processing frame: {frame_id}")
        except Exception as e:
            print(f"Error opening image {image_path}: {e}")
            return []

        detection_results = frame_data.get("detection_results", [])

        # Crop all bounding boxes in the frame
        for result in detection_results:
            bd_pts = result.get("bd_pts", [])
            if len(bd_pts) != 4:
                print(f"Invalid bounding box for frame {frame_id}: {bd_pts}")
                continue
            try:
                cropped_image = crop_image_in_memory(image, bd_pts)
                images.append(cropped_image)
                print(f"Cropped image for frame {frame_id}.")
            except Exception as e:
                print(f"Error cropping image {frame_id}: {e}")
                continue

    return images

def process_batch(predictor, batch_data):
    batch_images = []
    for frame_data in batch_data:
        images = process_frames(predictor, frame_data, image_dir)
        batch_images.extend(images)
    return batch_images

def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, batch_size=16):
    """Process images by reading bounding box data from a JSON file."""
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Initialize the predictor with the checkpoint
    predictor = PARSeqPredictor(checkpoint_path)

    # Extract all frames from the JSON file
    frame_ids = list(data.keys())
    frame_data_list = [{'frame_id': frame_id, 'detection_results': data[frame_id].get('detection_results', [])} for frame_id in frame_ids]

    results = []

    # Use ThreadPoolExecutor to process multiple batches concurrently
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for i in tqdm(range(0, len(frame_data_list), batch_size), desc="Processing Batches"):
            batch_data = frame_data_list[i:i + batch_size]
            print(f"Starting batch {i // batch_size + 1} with {len(batch_data)} frames.")
            futures.append(executor.submit(process_batch, predictor, batch_data))

        # Wait for all futures to complete and handle results
        for future in as_completed(futures):
            try:
                batch_images = future.result()
                if batch_images:
                    print(f"Batch completed: {len(batch_images)} images processed.")
                    start_pred_time = time.time()
                    # Predict for cropped images
                    pred_texts, confidences = predictor.predict(batch_images)
                    pred_time = time.time() - start_pred_time
                    print(f"Time taken for prediction on batch: {pred_time:.2f} seconds")
                    results.extend(zip(pred_texts, confidences))
            except Exception as e:
                print(f"Error in thread: {e}")

    # Print results
    if not results:
        print("No predictions were made.")
    else:
        for pred_text, confidence in results:
            print(f"Predicted Text: {pred_text}, Confidence: {confidence:.4f}")

    print(f"Total frames processed: {len(frame_data_list)}")
    print(f"Total predictions made: {len(results)}")

# Execute the program
if __name__ == "__main__":
    json_file = '/content/drive/MyDrive/L07_new/results/L07_V002_keyframes_filtered.json'
    image_dir = '/content/drive/MyDrive/L07_new/L07_V002_keyframes_filtered'
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, batch_size=16)
    end_time = time.time()

    print(f"Tổng thời gian xử lý: {end_time - start_time:.2f} giây")


Processing Batches: 100%|██████████| 56/56 [00:00<00:00, 1480.82it/s]


Streaming output truncated to the last 5000 lines.
Predicted Text: ['phân'], Confidence: 0.9965
Predicted Text: ['cách'], Confidence: 0.9991
Predicted Text: ['dải'], Confidence: 0.9979
Predicted Text: ['chiều'], Confidence: 0.9767
Predicted Text: ['cầu'], Confidence: 0.9976
Predicted Text: ['đầu'], Confidence: 0.9991
Predicted Text: ['đi'], Confidence: 0.9997
Predicted Text: ['cứng'], Confidence: 0.9854
Predicted Text: ['06:33:29'], Confidence: 0.9997
Predicted Text: ['ngược'], Confidence: 0.9945
Predicted Text: ['TP.'], Confidence: 0.9999
Predicted Text: ['Gian'], Confidence: 0.9999
Predicted Text: ['xe'], Confidence: 0.9998
Predicted Text: ['Bắc'], Confidence: 0.9881
Predicted Text: ['cách'], Confidence: 0.9989
Predicted Text: ['đuối'], Confidence: 0.8810
Predicted Text: ['chấp'], Confidence: 0.9981
Predicted Text: ['đi'], Confidence: 0.9999
Predicted Text: ['nối'], Confidence: 0.9994
Predicted Text: ['tải'], Confidence: 0.9991
Predicted Text: ['phân'], Confidence: 0.9976
Predicted T

In [ ]:
import os
import json
import time
from PIL import Image
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint
from tqdm import tqdm

# Class để load và predict ảnh sử dụng PARSeq
class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
        img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
        return parseq, img_transform

    @torch.inference_mode()
    def predict_batch(self, images):
        """Dự đoán một batch hình ảnh."""
        if not images:
            return [], 0.0
        try:
            images_tensor = torch.stack([self.img_transform(img).unsqueeze(0) for img in images]).to(self.device)
            p = self.parseq(images_tensor).softmax(-1)
            decoded = self.parseq.tokenizer.decode(p)

            # In ra giá trị decode để kiểm tra
            print(f"Decoded output: {decoded}")

            # Giả sử decode trả về 4 giá trị
            if len(decoded) == 4:
                preds, probs, value3, value4 = decoded
            elif len(decoded) >= 2:
                preds, probs, *rest = decoded
            else:
                raise ValueError(f"Hàm decode trả về {len(decoded)} giá trị, không được hỗ trợ.")

            # Tính toán độ tin cậy trung bình
            confidence = torch.mean(probs).item() if isinstance(probs, torch.Tensor) else 0.0
            return preds, confidence
        except Exception as e:
            print(f"Lỗi trong predict_batch: {e}")
            return [], 0.0

# Hàm để crop ảnh dựa trên bounding box
def crop_image_in_memory(image, bd_pts):
    """Crop image in memory based trên bounding box."""
    try:
        left = bd_pts[0][0]
        top = bd_pts[0][1]
        right = bd_pts[2][0]
        bottom = bd_pts[2][1]
        cropped_image = image.crop((left, top, right, bottom))
        return cropped_image
    except Exception as e:
        print(f"Lỗi khi crop ảnh với bounding box {bd_pts}: {e}")
        return None

# Hàm xử lý một batch các frame ảnh
def process_frames(predictor, frame_data_list, image_dir):
    images = []
    detection_info = []  # Để theo dõi frame và bounding box tương ứng
    total_cropped_images = 0  # Biến đếm số lượng hình ảnh đã crop

    # Lấy hình ảnh và bounding box cho tất cả các frame
    for frame_data in frame_data_list:
        frame_id = frame_data['frame_id']
        image_path = os.path.join(image_dir, frame_id)
        if os.path.exists(image_path):
            try:
                image = Image.open(image_path).convert('RGB')
            except Exception as e:
                print(f"Lỗi khi mở ảnh {image_path}: {e}")
                continue
            detection_results = frame_data.get("detection_results", [])

            # Crop tất cả các bounding box trong frame
            for result in detection_results:
                bd_pts = result.get("bd_pts", [])
                if len(bd_pts) != 4:
                    print(f"Bounding box không hợp lệ cho frame {frame_id}: {bd_pts}")
                    continue
                cropped_image = crop_image_in_memory(image, bd_pts)
                if cropped_image:
                    images.append(cropped_image)
                    detection_info.append((frame_id, bd_pts))
                    total_cropped_images += 1

    # Dự đoán cho tất cả các hình ảnh trong batch
    if images:
        preds, confidence = predictor.predict_batch(images)
        if preds:
            for (frame_id, bd_pts), pred in zip(detection_info, preds):
                print(f"Frame ID: {frame_id}, Bounding Box: {bd_pts}, Predicted Text: {pred}, Confidence: {confidence:.4f}")
        else:
            print("Không có dự đoán nào được thực hiện.")
    else:
        print("Không có hình ảnh nào để xử lý trong batch này.")

    print(f"Tổng số hình ảnh đã crop trong batch này: {total_cropped_images}")

# Hàm xử lý ảnh dựa trên file JSON
def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, batch_size=16):
    """Xử lý ảnh bằng cách đọc dữ liệu bounding box từ file JSON."""
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Lỗi khi mở file JSON {json_file}: {e}")
        return

    # Khởi tạo predictor với checkpoint
    try:
        predictor = PARSeqPredictor(checkpoint_path)
    except Exception as e:
        print(f"Lỗi khi khởi tạo PARSeqPredictor: {e}")
        return

    # Lấy tất cả frame từ file JSON
    frame_ids = list(data.keys())
    frame_data_list = [{'frame_id': frame_id, 'detection_results': data[frame_id].get('detection_results', [])} for frame_id in frame_ids]

    # Sử dụng ThreadPoolExecutor để xử lý nhiều batch cùng lúc
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for i in tqdm(range(0, len(frame_data_list), batch_size), desc="Đang xử lý các batch"):
            batch_data = frame_data_list[i:i + batch_size]
            futures.append(executor.submit(process_frames, predictor, batch_data, image_dir))

        # Chờ tất cả các futures hoàn thành và xử lý ngoại lệ
        for future in as_completed(futures):
            try:
                future.result()
            except Exception as e:
                print(f"Lỗi trong thread: {e}")

# Thực thi chương trình
if __name__ == "__main__":
    json_file = '/content/drive/MyDrive/L07_new/results/L07_V002_keyframes_filtered.json'
    image_dir = '/content/drive/MyDrive/L07_new/L07_V002_keyframes_filtered'
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, batch_size=4)
    end_time = time.time()

    print(f"Tổng thời gian xử lý: {end_time - start_time:.2f} giây")


Đang xử lý các batch: 100%|██████████| 224/224 [00:00<00:00, 16985.29it/s]


Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 62
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 87
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 118
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 129
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 89
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 65
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã c

In [ ]:
import os
import json
import time
from PIL import Image
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint
from tqdm import tqdm

# Class để load và predict ảnh sử dụng PARSeq
class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
        img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
        return parseq, img_transform

    @torch.inference_mode()
    def predict_batch(self, images):
        """Dự đoán một batch hình ảnh."""
        if not images:
            return [], 0.0
        try:
            images_tensor = torch.stack([self.img_transform(img).unsqueeze(0) for img in images]).to(self.device)
            p = self.parseq(images_tensor).softmax(-1)
            decoded = self.parseq.tokenizer.decode(p)

            # In ra giá trị decode để kiểm tra
            print(f"Decoded output: {decoded}")
            print(f"Decoded output type: {type(decoded)}")
            print(f"Length of decoded output: {len(decoded) if isinstance(decoded, (list, tuple)) else 'N/A'}")

            preds = []
            probs = []

            if isinstance(decoded, list) or isinstance(decoded, tuple):
                for item in decoded:
                    if isinstance(item, (list, tuple)):
                        if len(item) >= 2:
                            preds.append(item[0])
                            probs.append(item[1])
                        else:
                            preds.append(str(item))
                            probs.append(0.0)
                    else:
                        preds.append(str(item))
                        probs.append(0.0)
            else:
                # Nếu decoded không phải là list hoặc tuple
                preds.append(str(decoded))
                probs.append(0.0)

            # Tính toán độ tin cậy trung bình
            confidence = torch.mean(torch.tensor(probs)).item() if probs else 0.0
            return preds, confidence
        except Exception as e:
            print(f"Lỗi trong predict_batch: {e}")
            return [], 0.0

# Hàm để crop ảnh dựa trên bounding box
def crop_image_in_memory(image, bd_pts):
    """Crop image in memory dựa trên bounding box."""
    try:
        left = bd_pts[0][0]
        top = bd_pts[0][1]
        right = bd_pts[2][0]
        bottom = bd_pts[2][1]
        cropped_image = image.crop((left, top, right, bottom))
        return cropped_image
    except Exception as e:
        print(f"Lỗi khi crop ảnh với bounding box {bd_pts}: {e}")
        return None

# Hàm xử lý một batch các frame ảnh
def process_frames(predictor, frame_data_list, image_dir):
    images = []
    detection_info = []  # Để theo dõi frame và bounding box tương ứng
    total_cropped_images = 0  # Biến đếm số lượng hình ảnh đã crop

    # Lấy hình ảnh và bounding box cho tất cả các frame
    for frame_data in frame_data_list:
        frame_id = frame_data['frame_id']
        image_path = os.path.join(image_dir, frame_id)
        if os.path.exists(image_path):
            try:
                image = Image.open(image_path).convert('RGB')
            except Exception as e:
                print(f"Lỗi khi mở ảnh {image_path}: {e}")
                continue
            detection_results = frame_data.get("detection_results", [])

            # Crop tất cả các bounding box trong frame
            for result in detection_results:
                bd_pts = result.get("bd_pts", [])
                if len(bd_pts) != 4:
                    print(f"Bounding box không hợp lệ cho frame {frame_id}: {bd_pts}")
                    continue
                cropped_image = crop_image_in_memory(image, bd_pts)
                if cropped_image:
                    images.append(cropped_image)
                    detection_info.append((frame_id, bd_pts))
                    total_cropped_images += 1

    # Dự đoán cho tất cả các hình ảnh trong batch
    if images:
        preds, confidence = predictor.predict_batch(images)
        if preds:
            for (frame_id, bd_pts), pred in zip(detection_info, preds):
                print(f"Frame ID: {frame_id}, Bounding Box: {bd_pts}, Predicted Text: {pred}, Confidence: {confidence:.4f}")
        else:
            print("Không có dự đoán nào được thực hiện.")
    else:
        print("Không có hình ảnh nào để xử lý trong batch này.")

    print(f"Tổng số hình ảnh đã crop trong batch này: {total_cropped_images}")

# Hàm xử lý ảnh dựa trên file JSON
def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, batch_size=16):
    """Xử lý ảnh bằng cách đọc dữ liệu bounding box từ file JSON."""
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Lỗi khi mở file JSON {json_file}: {e}")
        return

    # Khởi tạo predictor với checkpoint
    try:
        predictor = PARSeqPredictor(checkpoint_path)
    except Exception as e:
        print(f"Lỗi khi khởi tạo PARSeqPredictor: {e}")
        return

    # Lấy tất cả frame từ file JSON
    frame_ids = list(data.keys())
    frame_data_list = [{'frame_id': frame_id, 'detection_results': data[frame_id].get('detection_results', [])} for frame_id in frame_ids]

    # Sử dụng ThreadPoolExecutor để xử lý nhiều batch cùng lúc
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for i in tqdm(range(0, len(frame_data_list), batch_size), desc="Đang xử lý các batch"):
            batch_data = frame_data_list[i:i + batch_size]
            futures.append(executor.submit(process_frames, predictor, batch_data, image_dir))

        # Chờ tất cả các futures hoàn thành và xử lý ngoại lệ
        for future in as_completed(futures):
            try:
                future.result()
            except Exception as e:
                print(f"Lỗi trong thread: {e}")

# Thực thi chương trình
if __name__ == "__main__":
    json_file = '/content/drive/MyDrive/L07_new/results/L07_V002_keyframes_filtered.json'
    image_dir = '/content/drive/MyDrive/L07_new/L07_V002_keyframes_filtered'
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, batch_size=16)
    end_time = time.time()

    print(f"Tổng thời gian xử lý: {end_time - start_time:.2f} giây")


Đang xử lý các batch: 100%|██████████| 56/56 [00:00<00:00, 10299.54it/s]


Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 358
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 396
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 442
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 407
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 129
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh đã crop trong batch này: 182
Lỗi trong predict_batch: too many values to unpack (expected 4)
Không có dự đoán nào được thực hiện.
Tổng số hình ảnh 

In [ ]:
import os
import json
import time
import re
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint
import torch

class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
        img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
        return parseq, img_transform

    @torch.inference_mode()
    def predict(self, image):
        image = self.img_transform(image).unsqueeze(0).to(self.device)
        p = self.parseq(image).softmax(-1)
        pred, p = self.parseq.tokenizer.decode(p)
        return pred, torch.mean(p[0])

def crop_image_in_memory(image, bd_pts):
    left, top = bd_pts[0]
    right, bottom = bd_pts[2]
    return image.crop((left, top, right, bottom))


def process_frame(predictor, frame_id, frame_data, image_dir):
    image_path = os.path.join(image_dir, frame_id)
    if os.path.exists(image_path):
        image = Image.open(image_path)
        detection_results = frame_data["detection_results"]
        frame_texts = []
        for result in detection_results:
            bd_pts = result["bd_pts"]
            cropped_image = crop_image_in_memory(image, bd_pts)
            pred_text, confidence = predictor.predict(cropped_image)
            if confidence > 0.5:
                frame_texts.append(pred_text)
        return frame_id, ' '.join(frame_texts)
    else:
        print(f"Image {frame_id} not found in directory {image_dir}")
        return frame_id, ""

def process_video(json_file, image_dir, checkpoint_path, max_workers=4):
    with open(json_file, 'r') as f:
        data = json.load(f)

    predictor = PARSeqPredictor(checkpoint_path)

    video_results = {}

    with tqdm(total=len(data), desc="Processing frames", unit="frame") as pbar:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(process_frame, predictor, frame_id, frame_data, image_dir)
                       for frame_id, frame_data in data.items()]

            for future in futures:
                frame_id, frame_text = future.result()
                if frame_text:
                    video_results[frame_id] = frame_text
                pbar.update(1)

    return video_results

def process_folder(folder_path, results_path, checkpoint_path):
    ocr_results = {"OCR": {}}

    json_files = [f for f in os.listdir(results_path) if f.endswith('.json')]

    with tqdm(total=len(json_files), desc="Processing videos", unit="video") as pbar:
        for json_file in json_files:
            video_name = json_file.replace('_keyframes_filtered.json', '')
            json_file_path = os.path.join(results_path, json_file)
            image_dir = os.path.join(folder_path, f"{video_name}_keyframes_filtered")

            print(f"Processing {video_name}")
            print(f"JSON file: {json_file_path}")
            print(f"Image directory: {image_dir}")

            if os.path.exists(json_file_path) and os.path.exists(image_dir):
                video_results = process_video(json_file_path, image_dir, checkpoint_path)
                ocr_results["OCR"][video_name] = video_results
            else:
                print(f"Missing JSON file or image directory for {video_name}")

            pbar.update(1)

    return ocr_results

if __name__ == "__main__":
    folder_path = '/content/drive/MyDrive/L07_new'  # Path to L07_new folder
    results_path = '/content/drive/MyDrive/L07_new/results'  # Path to results folder
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'
    output_json_path = '/content/ocr_results_L07_1.json'

    start_time = time.time()
    ocr_results = process_folder(folder_path, results_path, checkpoint_path)
    end_time = time.time()

    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(ocr_results, f, ensure_ascii=False, indent=2)

    print(f"OCR results saved to {output_json_path}")
    print(f"Total processing time: {end_time - start_time} seconds")

Processing videos:   0%|          | 0/62 [00:00<?, ?video/s]

Processing L07_V001
JSON file: /content/drive/MyDrive/L07_new/results/L07_V001_keyframes_filtered.json
Image directory: /content/drive/MyDrive/L07_new/L07_V001_keyframes_filtered



Processing videos:   0%|          | 0/62 [00:44<?, ?video/s]
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "<ipython-input-10-c04750217371>", line 66, in process_video
    frame_id, frame_text = future.result()
  File "/usr/lib/python3.10/concurrent/futures/_base.py", line 458, in result
    return self.__get_result()
  File "/usr/lib/python3.10/concurrent/futures/_base.py", line 403, in __get_result
    raise self._exception
  File "/usr/lib/python3.10/concurrent/futures/thread.py", line 58, in run
    result = self.fn(*self.args, **self.kwargs)
  File "<ipython-input-10-c04750217371>", line 47, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-10-c04750217371>", line 105, in <cell line: 98>
   

TypeError: object of type 'NoneType' has no len()

In [ ]:
import os
import json
import time
import re
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint
import torch
import traceback

# Class để load và predict ảnh sử dụng PARSeq
class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        try:
            parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
            img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
            return parseq, img_transform
        except Exception as e:
            print(f"Error loading model: {str(e)}")
            raise

    @torch.inference_mode()
    def predict(self, image):
        try:
            image = self.img_transform(image).unsqueeze(0).to(self.device)
            p = self.parseq(image).softmax(-1)
            pred, p = self.parseq.tokenizer.decode(p)
            return pred, torch.mean(p[0])
        except Exception as e:
            print(f"Error in prediction: {str(e)}")
            raise

# Hàm crop ảnh dựa trên tọa độ bounding box
def crop_image_in_memory(image, bd_pts):
    try:
        left, top = bd_pts[0]
        right, bottom = bd_pts[2]
        return image.crop((left, top, right, bottom))
    except Exception as e:
        print(f"Error cropping image: {str(e)}")
        raise

# Hàm xử lý từng frame và nhận diện văn bản
def process_frame(predictor, frame_id, frame_data, image_dir):
    try:
        image_path = os.path.join(image_dir, frame_id)
        if os.path.exists(image_path):
            image = Image.open(image_path)
            detection_results = frame_data["detection_results"]
            frame_texts = []
            for result in detection_results:
                bd_pts = result["bd_pts"]
                cropped_image = crop_image_in_memory(image, bd_pts)
                pred_text, confidence = predictor.predict(cropped_image)

                # Ensure the predicted text is handled correctly
                if confidence > 0.5:
                    if isinstance(pred_text, list):
                        # If prediction is a list, flatten the list and join into a single string
                        flat_text = ' '.join([str(item) for item in pred_text])
                        frame_texts.append(flat_text)
                    elif isinstance(pred_text, str):
                        frame_texts.append(pred_text)
                    else:
                        # Convert non-string prediction to string (if it's a different type)
                        frame_texts.append(str(pred_text))

            # Ensure all items in frame_texts are strings before joining
            frame_texts = [str(text) for text in frame_texts if isinstance(text, (str, int, float))]
            return frame_id, ' '.join(frame_texts)  # Join all texts into a single string
        else:
            print(f"Image {frame_id} not found in directory {image_dir}")
            return frame_id, ""
    except Exception as e:
        print(f"Error processing frame {frame_id}: {str(e)}")
        traceback.print_exc()
        return frame_id, ""

# Hàm xử lý tất cả các khung hình từ một video dựa trên file JSON
def process_video(json_file, image_dir, checkpoint_path, max_workers=4):
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)

        predictor = PARSeqPredictor(checkpoint_path)

        video_results = {}

        with tqdm(total=len(data), desc="Processing frames", unit="frame") as pbar:
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = [executor.submit(process_frame, predictor, frame_id, frame_data, image_dir)
                           for frame_id, frame_data in data.items()]

                for future in futures:
                    frame_id, frame_text = future.result()
                    if frame_text:
                        video_results[frame_id] = frame_text
                    pbar.update(1)

        return video_results
    except Exception as e:
        print(f"Error processing video: {str(e)}")
        traceback.print_exc()
        return {}

# Hàm xử lý toàn bộ thư mục chứa các file video JSON và ảnh
def process_folder(folder_path, results_path, checkpoint_path):
    ocr_results = {"OCR": {}}

    json_files = [f for f in os.listdir(results_path) if f.endswith('.json')]

    with tqdm(total=len(json_files), desc="Processing videos", unit="video") as pbar:
        for json_file in json_files:
            try:
                video_name = json_file.replace('_keyframes_filtered.json', '')
                json_file_path = os.path.join(results_path, json_file)
                image_dir = os.path.join(folder_path, f"{video_name}_keyframes_filtered")

                print(f"Processing {video_name}")
                print(f"JSON file: {json_file_path}")
                print(f"Image directory: {image_dir}")

                if os.path.exists(json_file_path) and os.path.exists(image_dir):
                    video_results = process_video(json_file_path, image_dir, checkpoint_path)
                    ocr_results["OCR"][video_name] = video_results
                else:
                    print(f"Missing JSON file or image directory for {video_name}")

                pbar.update(1)
            except Exception as e:
                print(f"Error processing {json_file}: {str(e)}")
                traceback.print_exc()

    return ocr_results

# Thực thi chương trình
if __name__ == "__main__":
    folder_path = '/content/drive/MyDrive/L07_new'  # Đường dẫn tới thư mục chứa hình ảnh
    results_path = '/content/drive/MyDrive/L07_new/results'  # Đường dẫn tới thư mục chứa file JSON
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'  # Đường dẫn tới mô hình checkpoint
    output_json_path = '/content/ocr_results_L07_1.json'  # Đường dẫn lưu kết quả OCR

    start_time = time.time()
    try:
        ocr_results = process_folder(folder_path, results_path, checkpoint_path)
        end_time = time.time()

        # Lưu kết quả OCR vào file JSON
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(ocr_results, f, ensure_ascii=False, indent=2)

        print(f"OCR results saved to {output_json_path}")
        print(f"Total processing time: {end_time - start_time} seconds")
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        traceback.print_exc()


Predicted text: ['dùng'], Confidence: 0.9988, type: <class 'list'>
Error processing frame frame_19379.jpg: sequence item 0: expected str instance, list found


Traceback (most recent call last):
  File "<ipython-input-12-9065684566fe>", line 63, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found
Processing videos:   0%|          | 0/62 [00:00<?, ?video/s]

Processing L07_V001
JSON file: /content/drive/MyDrive/L07_new/results/L07_V001_keyframes_filtered.json
Image directory: /content/drive/MyDrive/L07_new/L07_V001_keyframes_filtered
Predicted text: ['trong'], Confidence: 0.9994, type: <class 'list'>
Error processing frame frame_23225.jpg: sequence item 0: expected str instance, list found
Predicted text: ['tư'], Confidence: 0.9975, type: <class 'list'>


Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found


Predicted text: ['tiếp'], Confidence: 0.9980, type: <class 'list'>
Predicted text: ['GÁI'], Confidence: 0.9997, type: <class 'list'>
Predicted text: ['trực'], Confidence: 0.9977, type: <class 'list'>
Predicted text: ['CỬU'], Confidence: 0.9885, type: <class 'list'>
Predicted text: ['Vốn'], Confidence: 0.9894, type: <class 'list'>
Error processing frame frame_23443.jpg: sequence item 0: expected str instance, list found


...

Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found


Predicted text: ['06:45:25'], Confidence: 0.9999, type: <class 'list'>
Predicted text: ['SUBSCRIBE'], Confidence: 0.9706, type: <class 'list'>
Predicted text: ['Đà'], Confidence: 0.9998, type: <class 'list'>
Predicted text: ['Khánh'], Confidence: 0.9996, type: <class 'list'>
Predicted text: ['HD'], Confidence: 0.9997, type: <class 'list'>


Processing videos:   0%|          | 0/62 [03:21<?, ?video/s]

Predicted text: ['06:45:27'], Confidence: 0.9999, type: <class 'list'>
Predicted text: ['hàng'], Confidence: 0.9995, type: <class 'list'>


KeyboardInterrupt: 

In [ ]:
import os
import json
import time
import re
from PIL import Image
import torch
import torch.multiprocessing as mp
from tqdm import tqdm
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint
import traceback

# Class để load và predict ảnh sử dụng PARSeq
class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path)

    def load_model_parseq(self, checkpoint_path):
        try:
            parseq = load_from_checkpoint(checkpoint_path).eval().to(self.device)
            img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
            return parseq, img_transform
        except Exception as e:
            print(f"Error loading model: {str(e)}")
            raise

    @torch.inference_mode()
    def predict(self, image):
        try:
            image = self.img_transform(image).unsqueeze(0).to(self.device)
            p = self.parseq(image).softmax(-1)
            pred, p = self.parseq.tokenizer.decode(p)
            return pred, torch.mean(p[0])
        except Exception as e:
            print(f"Error in prediction: {str(e)}")
            raise

# Hàm crop ảnh dựa trên tọa độ bounding box
def crop_image_in_memory(image, bd_pts):
    try:
        left, top = bd_pts[0]
        right, bottom = bd_pts[2]
        return image.crop((left, top, right, bottom))
    except Exception as e:
        print(f"Error cropping image: {str(e)}")
        raise

# Hàm xử lý từng frame và nhận diện văn bản
def process_frame(predictor, frame_id, frame_data, image_dir):
    try:
        image_path = os.path.join(image_dir, frame_id)
        if os.path.exists(image_path):
            image = Image.open(image_path)
            detection_results = frame_data["detection_results"]
            frame_texts = []
            for result in detection_results:
                bd_pts = result["bd_pts"]
                cropped_image = crop_image_in_memory(image, bd_pts)
                pred_text, confidence = predictor.predict(cropped_image)

                # Ensure the predicted text is handled correctly
                if confidence > 0.5:
                    if isinstance(pred_text, list):
                        # If prediction is a list, flatten the list and join into a single string
                        flat_text = ' '.join([str(item) for item in pred_text])
                        frame_texts.append(flat_text)
                    elif isinstance(pred_text, str):
                        frame_texts.append(pred_text)
                    else:
                        # Convert non-string prediction to string (if it's a different type)
                        frame_texts.append(str(pred_text))

            # Ensure all items in frame_texts are strings before joining
            frame_texts = [str(text) for text in frame_texts if isinstance(text, (str, int, float))]
            return frame_id, ' '.join(frame_texts)  # Join all texts into a single string
        else:
            print(f"Image {frame_id} not found in directory {image_dir}")
            return frame_id, ""
    except Exception as e:
        print(f"Error processing frame {frame_id}: {str(e)}")
        traceback.print_exc()
        return frame_id, ""

# Hàm để xử lý video với multiprocessing
def process_video_multiprocessing(json_file, image_dir, checkpoint_path, max_workers=4):
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)

        video_results = {}

        # Khởi tạo predictor trong mỗi tiến trình riêng
        predictor = PARSeqPredictor(checkpoint_path)

        # Sử dụng multiprocessing
        with mp.Pool(processes=max_workers) as pool:
            results = pool.starmap(process_frame, [(predictor, frame_id, frame_data, image_dir) for frame_id, frame_data in data.items()])

            for frame_id, frame_text in results:
                if frame_text:
                    video_results[frame_id] = frame_text

        return video_results
    except Exception as e:
        print(f"Error processing video: {str(e)}")
        traceback.print_exc()
        return {}

# Hàm xử lý toàn bộ thư mục chứa các file video JSON và ảnh
def process_folder(folder_path, results_path, checkpoint_path, max_workers=4):
    ocr_results = {"OCR": {}}

    json_files = [f for f in os.listdir(results_path) if f.endswith('.json')]

    with tqdm(total=len(json_files), desc="Processing videos", unit="video") as pbar:
        for json_file in json_files:
            try:
                video_name = json_file.replace('_keyframes_filtered.json', '')
                json_file_path = os.path.join(results_path, json_file)
                image_dir = os.path.join(folder_path, f"{video_name}_keyframes_filtered")

                print(f"Processing {video_name}")
                print(f"JSON file: {json_file_path}")
                print(f"Image directory: {image_dir}")

                if os.path.exists(json_file_path) and os.path.exists(image_dir):
                    video_results = process_video_multiprocessing(json_file_path, image_dir, checkpoint_path, max_workers)
                    ocr_results["OCR"][video_name] = video_results
                else:
                    print(f"Missing JSON file or image directory for {video_name}")

                pbar.update(1)
            except Exception as e:
                print(f"Error processing {json_file}: {str(e)}")
                traceback.print_exc()

    return ocr_results

# Thực thi chương trình
if __name__ == "__main__":
    mp.set_start_method('spawn', force=True)  # Sử dụng 'spawn' cho multiprocessing với CUDA

    folder_path = '/content/drive/MyDrive/L07_new'  # Đường dẫn tới thư mục chứa hình ảnh
    results_path = '/content/drive/MyDrive/L07_new/results'  # Đường dẫn tới thư mục chứa file JSON
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'  # Đường dẫn tới mô hình checkpoint
    output_json_path = '/content/ocr_results_L07_1.json'  # Đường dẫn lưu kết quả OCR

    start_time = time.time()
    try:
        ocr_results = process_folder(folder_path, results_path, checkpoint_path, max_workers=4)
        end_time = time.time()

        # Lưu kết quả OCR vào file JSON
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(ocr_results, f, ensure_ascii=False, indent=2)

        print(f"OCR results saved to {output_json_path}")
        print(f"Total processing time: {end_time - start_time} seconds")
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        traceback.print_exc()


Processing videos:   0%|          | 0/62 [00:00<?, ?video/s]

Processing L07_V001
JSON file: /content/drive/MyDrive/L07_new/results/L07_V001_keyframes_filtered.json
Image directory: /content/drive/MyDrive/L07_new/L07_V001_keyframes_filtered
Error processing frame frame_3199.jpg: sequence item 0: expected str instance, list found


Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found


Error processing frame frame_338.jpg: sequence item 0: expected str instance, list found
Error processing frame frame_3224.jpg: sequence item 0: expected str instance, list found


Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found
Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found
Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found


...

Traceback (most recent call last):
  File "<ipython-input-12-9065684566fe>", line 63, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found


Error processing frame frame_42.jpg: sequence item 0: expected str instance, list found


Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found
Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found
Processing videos:   0%|          | 0/62 [00:56<?, ?video/s]

Error processing frame frame_4161.jpg: sequence item 0: expected str instance, list found


KeyboardInterrupt: 

In [ ]:
import os
import json
import time
import re
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint
import torch
import traceback

# Class để load và predict ảnh sử dụng PARSeq
class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        try:
            parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
            img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
            return parseq, img_transform
        except Exception as e:
            print(f"Error loading model: {str(e)}")
            raise

    @torch.inference_mode()
    def predict(self, image):
        try:
            image = self.img_transform(image).unsqueeze(0).to(self.device)
            p = self.parseq(image).softmax(-1)
            pred, p = self.parseq.tokenizer.decode(p)
            return pred, torch.mean(p[0])
        except Exception as e:
            print(f"Error in prediction: {str(e)}")
            raise

# Hàm crop ảnh dựa trên tọa độ bounding box
def crop_image_in_memory(image, bd_pts):
    try:
        left, top = bd_pts[0]
        right, bottom = bd_pts[2]
        return image.crop((left, top, right, bottom))
    except Exception as e:
        print(f"Error cropping image: {str(e)}")
        raise

# Hàm xử lý từng frame và nhận diện văn bản
def process_frame(predictor, frame_id, frame_data, image_dir):
    try:
        image_path = os.path.join(image_dir, frame_id)
        if os.path.exists(image_path):
            image = Image.open(image_path)
            detection_results = frame_data["detection_results"]
            frame_texts = []
            for result in detection_results:
                bd_pts = result["bd_pts"]
                cropped_image = crop_image_in_memory(image, bd_pts)
                pred_text, confidence = predictor.predict(cropped_image)

                # Ensure the predicted text is a string
                if confidence > 0.5:
                    if isinstance(pred_text, str):
                        frame_texts.append(pred_text)
                    elif isinstance(pred_text, list):
                        # Convert list items to a single string, if prediction returns a list
                        frame_texts.append(' '.join([str(item) for item in pred_text]))
                    else:
                        # Convert non-string prediction to string (if it's a different type)
                        frame_texts.append(str(pred_text))

            return frame_id, ' '.join(frame_texts)  # Join all texts into a single string
        else:
            print(f"Image {frame_id} not found in directory {image_dir}")
            return frame_id, ""
    except Exception as e:
        print(f"Error processing frame {frame_id}: {str(e)}")
        traceback.print_exc()
        return frame_id, ""

# Hàm xử lý tất cả các khung hình từ một video dựa trên file JSON
def process_video(json_file, image_dir, checkpoint_path, max_workers=4):
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)

        predictor = PARSeqPredictor(checkpoint_path)

        video_results = {}

        with tqdm(total=len(data), desc="Processing frames", unit="frame") as pbar:
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                futures = [executor.submit(process_frame, predictor, frame_id, frame_data, image_dir)
                           for frame_id, frame_data in data.items()]

                for future in futures:
                    frame_id, frame_text = future.result()
                    if frame_text:
                        video_results[frame_id] = frame_text
                    pbar.update(1)

        return video_results
    except Exception as e:
        print(f"Error processing video: {str(e)}")
        traceback.print_exc()
        return {}

# Hàm xử lý toàn bộ thư mục chứa các file video JSON và ảnh
def process_folder(folder_path, results_path, checkpoint_path):
    ocr_results = {"OCR": {}}

    json_files = [f for f in os.listdir(results_path) if f.endswith('.json')]

    with tqdm(total=len(json_files), desc="Processing videos", unit="video") as pbar:
        for json_file in json_files:
            try:
                video_name = json_file.replace('_keyframes_filtered.json', '')
                json_file_path = os.path.join(results_path, json_file)
                image_dir = os.path.join(folder_path, f"{video_name}_keyframes_filtered")

                print(f"Processing {video_name}")
                print(f"JSON file: {json_file_path}")
                print(f"Image directory: {image_dir}")

                if os.path.exists(json_file_path) and os.path.exists(image_dir):
                    video_results = process_video(json_file_path, image_dir, checkpoint_path)
                    ocr_results["OCR"][video_name] = video_results
                else:
                    print(f"Missing JSON file or image directory for {video_name}")

                pbar.update(1)
            except Exception as e:
                print(f"Error processing {json_file}: {str(e)}")
                traceback.print_exc()

    return ocr_results

# Thực thi chương trình
if __name__ == "__main__":
    folder_path = '/content/drive/MyDrive/L07_new'  # Đường dẫn tới thư mục chứa hình ảnh
    results_path = '/content/drive/MyDrive/L07_new/results'  # Đường dẫn tới thư mục chứa file JSON
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'  # Đường dẫn tới mô hình checkpoint
    output_json_path = '/content/ocr_results_L07_1.json'  # Đường dẫn lưu kết quả OCR

    start_time = time.time()
    try:
        ocr_results = process_folder(folder_path, results_path, checkpoint_path)
        end_time = time.time()

        # Lưu kết quả OCR vào file JSON
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(ocr_results, f, ensure_ascii=False, indent=2)

        print(f"OCR results saved to {output_json_path}")
        print(f"Total processing time: {end_time - start_time} seconds")
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        traceback.print_exc()


Processing videos:   0%|          | 0/62 [00:00<?, ?video/s]

Processing L07_V001
JSON file: /content/drive/MyDrive/L07_new/results/L07_V001_keyframes_filtered.json
Image directory: /content/drive/MyDrive/L07_new/L07_V001_keyframes_filtered
Error processing frame frame_14787.jpg: sequence item 0: expected str instance, list found


Traceback (most recent call last):
  File "<ipython-input-12-9065684566fe>", line 63, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found
Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found


Error processing frame frame_19643.jpg: sequence item 0: expected str instance, list found



Processing frames:   0%|          | 0/735 [00:00<?, ?frame/s]

...

Error processing frame frame_1562.jpg: sequence item 0: expected str instance, list found


Traceback (most recent call last):
  File "<ipython-input-12-9065684566fe>", line 63, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found


Error processing frame frame_20057.jpg: sequence item 0: expected str instance, list found


Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found
Processing videos:   0%|          | 0/62 [00:42<?, ?video/s]


KeyboardInterrupt: 

In [ ]:
import torch
from PIL import Image
import sys
import statistics
import os
import json
import time
import traceback
from tqdm import tqdm

# Thêm đường dẫn tới thư viện PARSeq
sys.path.append('/content/Run_parseq_ocr/parseq')

from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint

# Lớp dự đoán PARSeq
class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
        img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
        return parseq, img_transform

    @torch.inference_mode()
    def predict(self, image_path):
        image = Image.open(image_path).convert("RGB")
        pred_text, confidence = self.predict_parseq(image)
        return pred_text, confidence

    @torch.inference_mode()
    def predict_parseq(self, image):
        image = self.img_transform(image).unsqueeze(0).to(self.device)
        p = self.parseq(image).softmax(-1)
        pred, p = self.parseq.tokenizer.decode(p)
        return (pred, statistics.mean(p[0].tolist()))

# Hàm xử lý thư mục ảnh
def process_folder(folder_path, results_path, checkpoint_path, device='cuda'):
    # Tạo thư mục lưu kết quả nếu chưa tồn tại
    os.makedirs(results_path, exist_ok=True)

    # Lấy danh sách tất cả các file ảnh trong thư mục
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]

    # Tạo từ điển để lưu kết quả OCR
    ocr_results = {'OCR': {}}

    # Tên của thư mục hiện tại
    folder_name = os.path.basename(folder_path)
    ocr_results['OCR'][folder_name] = {}

    # Khởi tạo Predictor
    predictor = PARSeqPredictor(checkpoint_path, device)

    # Duyệt qua từng file ảnh trong thư mục
    for image_file in tqdm(image_files, desc="Processing images"):
        image_path = os.path.join(folder_path, image_file)
        try:
            # Thực hiện OCR dự đoán
            pred_text, confidence = predictor.predict(image_path)

            # Lưu kết quả vào từ điển theo định dạng JSON yêu cầu
            ocr_results['OCR'][folder_name][image_file] = pred_text

        except Exception as e:
            print(f"Error processing {image_file}: {str(e)}")
            traceback.print_exc()

    return ocr_results

# Phần main để chạy chương trình
if __name__ == "__main__":
    folder_path = '/content/drive/MyDrive/L07_new'  # Đường dẫn tới thư mục chứa hình ảnh
    results_path = '/content/drive/MyDrive/L07_new/results'  # Đường dẫn tới thư mục chứa file JSON
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'  # Đường dẫn tới mô hình checkpoint
    output_json_path = '/content/ocr_results_L07_1.json'  # Đường dẫn lưu kết quả OCR

    start_time = time.time()

    try:
        # Gọi hàm xử lý thư mục ảnh và lấy kết quả OCR
        ocr_results = process_folder(folder_path, results_path, checkpoint_path)

        end_time = time.time()

        # Lưu kết quả OCR vào file JSON với định dạng mong muốn
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(ocr_results, f, ensure_ascii=False, indent=2)

        print(f"OCR results saved to {output_json_path}")
        print(f"Total processing time: {end_time - start_time} seconds")

    except Exception as e:
        print(f"An error occurred: {str(e)}")
        traceback.print_exc()


Error processing frame frame_9711.jpg: sequence item 0: expected str instance, list found


Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found


Error processing frame frame_9781.jpg: sequence item 0: expected str instance, list found


Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found
Processing images: 0it [00:00, ?it/s]

OCR results saved to /content/ocr_results_L07_1.json
Total processing time: 4.016775846481323 seconds



Traceback (most recent call last):
  File "<ipython-input-11-58fa5387eae2>", line 60, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found
Traceback (most recent call last):
  File "<ipython-input-12-9065684566fe>", line 63, in process_frame
    return frame_id, ' '.join(frame_texts)
TypeError: sequence item 0: expected str instance, list found


Error processing frame frame_9782.jpg: sequence item 0: expected str instance, list found
Error processing frame frame_4661.jpg: sequence item 0: expected str instance, list found


In [ ]:
import os
import json
import time
from PIL import Image
import torch
from concurrent.futures import ThreadPoolExecutor
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint
from tqdm import tqdm

# Class để load và predict ảnh sử dụng PARSeq
class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
        img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
        return parseq, img_transform

    @torch.inference_mode()
    def predict_batch(self, images):
        """Dự đoán một batch hình ảnh."""
        images_tensor = torch.stack([self.img_transform(img).unsqueeze(0) for img in images]).to(self.device)
        p = self.parseq(images_tensor).softmax(-1)
        preds, probs = self.parseq.tokenizer.decode(p)
        return preds, torch.mean(probs[0])

# Hàm để crop ảnh dựa trên bounding box
def crop_image_in_memory(image, bd_pts):
    """Crop image in memory based on the bounding box."""
    left = bd_pts[0][0]
    top = bd_pts[0][1]
    right = bd_pts[2][0]
    bottom = bd_pts[2][1]
    cropped_image = image.crop((left, top, right, bottom))
    return cropped_image

# Hàm xử lý một batch các frame ảnh
def process_frames(predictor, frame_data_list, image_dir):
    images = []
    frame_ids = []
    total_cropped_images = 0  # Biến đếm số lượng hình ảnh đã crop

    # Lấy hình ảnh và bounding box cho tất cả các frame
    for frame_data in frame_data_list:
        frame_id = frame_data['frame_id']
        frame_ids.append(frame_id)
        image_path = os.path.join(image_dir, frame_id)
        if os.path.exists(image_path):
            image = Image.open(image_path)
            detection_results = frame_data["detection_results"]

            # Crop tất cả các bounding box
            for result in detection_results:
                bd_pts = result["bd_pts"]
                cropped_image = crop_image_in_memory(image, bd_pts)
                images.append(cropped_image)
                total_cropped_images += 1  # Tăng biến đếm

    # Dự đoán cho tất cả các hình ảnh trong batch
    if images:
        preds, confidence = predictor.predict_batch(images)
        for idx, frame_id in enumerate(frame_ids):
            print(f"Frame ID: {frame_id}, Predicted text: {preds[idx]}, Confidence: {confidence:.4f}")

    print(f"Total cropped images in this batch: {total_cropped_images}")  # In ra số lượng hình ảnh đã crop

# Hàm xử lý ảnh dựa trên file JSON
def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=1, batch_size=16):
    """Process images by reading bounding box data from a JSON file."""
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Khởi tạo predictor với checkpoint
    predictor = PARSeqPredictor(checkpoint_path)

    # Lấy tất cả frame từ file JSON
    frame_ids = list(data.keys())
    frame_data_list = [{'frame_id': frame_id, 'detection_results': data[frame_id]['detection_results']} for frame_id in frame_ids]

    # Sử dụng ThreadPoolExecutor để xử lý nhiều batch cùng lúc
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for i in tqdm(range(0, len(frame_data_list), batch_size)):
            batch_data = frame_data_list[i:i + batch_size]
            executor.submit(process_frames, predictor, batch_data, image_dir)

# Thực thi chương trình
if __name__ == "__main__":
    json_file = '/content/drive/MyDrive/L07_new/results/L07_V002_keyframes_filtered.json'
    image_dir = '/content/drive/MyDrive/L07_new/L07_V002_keyframes_filtered'
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=1, batch_size=16)
    end_time = time.time()

    print(f"Tổng thời gian xử lý: {end_time - start_time} giây")


100%|██████████| 56/56 [00:00<00:00, 19346.10it/s]


Tổng thời gian xử lý: 16.041121006011963 giây


In [ ]:
import os
import json
import time
from PIL import Image
import torch
from concurrent.futures import ThreadPoolExecutor
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint
from tqdm import tqdm

# Class để load và predict ảnh sử dụng PARSeq
class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
        img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
        return parseq, img_transform

    @torch.inference_mode()
    def predict_batch(self, images):
        """Dự đoán một batch hình ảnh."""
        images_tensor = torch.stack([self.img_transform(img).unsqueeze(0) for img in images]).to(self.device)
        p = self.parseq(images_tensor).softmax(-1)
        preds, probs = self.parseq.tokenizer.decode(p)
        return preds, torch.mean(probs[0])

# Hàm để crop ảnh dựa trên bounding box
def crop_image_in_memory(image, bd_pts):
    """Crop image in memory based on the bounding box."""
    left = bd_pts[0][0]
    top = bd_pts[0][1]
    right = bd_pts[2][0]
    bottom = bd_pts[2][1]
    cropped_image = image.crop((left, top, right, bottom))
    return cropped_image

# Hàm xử lý một batch các frame ảnh
def process_frames(predictor, frame_data_list, image_dir):
    images = []
    frame_ids = []
    total_cropped_images = 0  # Biến đếm số lượng hình ảnh đã crop

    # Lấy hình ảnh và bounding box cho tất cả các frame
    for frame_data in frame_data_list:
        frame_id = frame_data['frame_id']
        frame_ids.append(frame_id)
        image_path = os.path.join(image_dir, frame_id)
        print(f"Processing frame: {frame_id}")  # In ra frame đang được xử lý

        if os.path.exists(image_path):
            image = Image.open(image_path)
            detection_results = frame_data["detection_results"]

            print(f"Found {len(detection_results)} detection results for {frame_id}.")  # Số lượng bounding box

            # Crop tất cả các bounding box
            for result in detection_results:
                bd_pts = result["bd_pts"]
                cropped_image = crop_image_in_memory(image, bd_pts)
                images.append(cropped_image)
                total_cropped_images += 1  # Tăng biến đếm

            # Dự đoán cho tất cả các hình ảnh trong batch
        if images:
            print(f"Predicting {len(images)} images in this batch...")  # In ra số lượng hình ảnh dự đoán
            preds, confidence = predictor.predict_batch(images)
            print(preds)
            # Kiểm tra số lượng preds và confidence
            print(f"Number of predictions: {len(preds)}")
            print(f"Number of confidences: {len(confidence)}")

            for idx, frame_id in enumerate(frame_ids):
                if idx < len(preds):  # Kiểm tra để tránh lỗi chỉ số
                    print(f"Frame ID: {frame_id}, Predicted text: {preds[idx]}, Confidence: {confidence:.4f}")
                else:
                    print(f"Warning: No prediction for Frame ID: {frame_id}")

        print(f"Total cropped images in this batch: {total_cropped_images}")  # In ra số lượng hình ảnh đã crop


# Hàm xử lý ảnh dựa trên file JSON
def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=8, batch_size=16):
    """Process images by reading bounding box data from a JSON file."""
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Khởi tạo predictor với checkpoint
    predictor = PARSeqPredictor(checkpoint_path)

    # Lấy tất cả frame từ file JSON
    frame_ids = list(data.keys())
    frame_data_list = [{'frame_id': frame_id, 'detection_results': data[frame_id]['detection_results']} for frame_id in frame_ids]

    # Sử dụng ThreadPoolExecutor để xử lý nhiều batch cùng lúc
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for i in tqdm(range(0, len(frame_data_list), batch_size)):
            batch_data = frame_data_list[i:i + batch_size]
            executor.submit(process_frames, predictor, batch_data, image_dir)

# Thực thi chương trình
if __name__ == "__main__":
    json_file = '/content/drive/MyDrive/L07_new/results/L07_V002_keyframes_filtered.json'
    image_dir = '/content/drive/MyDrive/L07_new/L07_V002_keyframes_filtered'
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=8, batch_size=16)
    end_time = time.time()

    print(f"Tổng thời gian xử lý: {end_time - start_time} giây")


100%|██████████| 56/56 [00:00<00:00, 5939.44it/s]


Processing frame: frame_0.jpg
Processing frame: frame_10784.jpg
Processing frame: frame_11933.jpg
Processing frame: frame_13395.jpg
Processing frame: frame_13874.jpg
Processing frame: frame_14236.jpg
Processing frame: frame_14516.jpg
Processing frame: frame_15243.jpg
Found 27 detection results for frame_10784.jpg.
Found 5 detection results for frame_13874.jpg.
Found 37 detection results for frame_13395.jpg.
Found 1 detection results for frame_14236.jpg.
Found 2 detection results for frame_0.jpg.
Predicting 5 images in this batch...
Found 22 detection results for frame_11933.jpg.
Predicting 1 images in this batch...
Processing frame: frame_15621.jpg
Predicting 27 images in this batch...
Processing frame: frame_15982.jpg
Found 11 detection results for frame_14516.jpg.
Found 31 detection results for frame_15243.jpg.
Predicting 2 images in this batch...
Processing frame: frame_16372.jpg
Predicting 37 images in this batch...
Predicting 22 images in this batch...
Found 22 detection results f

In [ ]:
import os
import json
import time
from PIL import Image
import torch
from concurrent.futures import ThreadPoolExecutor
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint
from tqdm import tqdm

# Class để load và predict ảnh sử dụng PARSeq
class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
        img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
        return parseq, img_transform

    @torch.inference_mode()
    def predict_batch(self, images):
        """Dự đoán một batch hình ảnh."""
        images_tensor = torch.stack([self.img_transform(img).unsqueeze(0) for img in images]).to(self.device)
        p = self.parseq(images_tensor).softmax(-1)
        preds, probs = self.parseq.tokenizer.decode(p)
        return preds, torch.mean(probs[0])

# Hàm để crop ảnh dựa trên bounding box
def crop_image_in_memory(image, bd_pts):
    """Crop image in memory based on the bounding box."""
    left = bd_pts[0][0]
    top = bd_pts[0][1]
    right = bd_pts[2][0]
    bottom = bd_pts[2][1]
    cropped_image = image.crop((left, top, right, bottom))
    return cropped_image

# Hàm xử lý một batch các frame ảnh
def process_frames(predictor, frame_data_list, image_dir):
    images = []
    frame_ids = []
    total_cropped_images = 0  # Biến đếm số lượng hình ảnh đã crop

    # Lấy hình ảnh và bounding box cho tất cả các frame
    for frame_data in frame_data_list:
        frame_id = frame_data['frame_id']
        frame_ids.append(frame_id)
        image_path = os.path.join(image_dir, frame_id)
        print(f"Processing frame: {frame_id}")  # In ra frame đang được xử lý

        if os.path.exists(image_path):
            image = Image.open(image_path)
            detection_results = frame_data["detection_results"]

            print(f"Found {len(detection_results)} detection results for {frame_id}.")  # Số lượng bounding box

            # Crop tất cả các bounding box
            for result in detection_results:
                bd_pts = result["bd_pts"]
                cropped_image = crop_image_in_memory(image, bd_pts)
                images.append(cropped_image)
                total_cropped_images += 1  # Tăng biến đếm

    # Dự đoán cho tất cả các hình ảnh trong batch
    if images:
        print(f"Predicting {len(images)} images in this batch...")  # In ra số lượng hình ảnh dự đoán
        preds, confidence = predictor.predict_batch(images)
        for idx, frame_id in enumerate(frame_ids):
            print(f"Frame ID: {frame_id}, Predicted text: {preds[idx]}, Confidence: {confidence:.4f}")

    print(f"Total cropped images in this batch: {total_cropped_images}")  # In ra số lượng hình ảnh đã crop

# Hàm xử lý ảnh dựa trên file JSON
def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=8, batch_size=16):
    """Process images by reading bounding box data from a JSON file."""
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Khởi tạo predictor với checkpoint
    predictor = PARSeqPredictor(checkpoint_path)

    # Lấy tất cả frame từ file JSON
    frame_ids = list(data.keys())
    frame_data_list = [{'frame_id': frame_id, 'detection_results': data[frame_id]['detection_results']} for frame_id in frame_ids]

    # Sử dụng ThreadPoolExecutor để xử lý nhiều batch cùng lúc
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for i in tqdm(range(0, len(frame_data_list), batch_size)):
            batch_data = frame_data_list[i:i + batch_size]
            executor.submit(process_frames, predictor, batch_data, image_dir)

# Thực thi chương trình
if __name__ == "__main__":
    json_file = '/content/drive/MyDrive/L07_new/results/L07_V002_keyframes_filtered.json'
    image_dir = '/content/drive/MyDrive/L07_new/L07_V002_keyframes_filtered'
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=8, batch_size=16)
    end_time = time.time()

    print(f"Tổng thời gian xử lý: {end_time - start_time} giây")


100%|██████████| 56/56 [00:00<00:00, 3742.41it/s]


Processing frame: frame_0.jpg
Processing frame: frame_10784.jpg
Processing frame: frame_11933.jpg
Processing frame: frame_13395.jpg
Processing frame: frame_13874.jpg
Processing frame: frame_14236.jpg
Processing frame: frame_14516.jpg
Processing frame: frame_15243.jpg
Found 2 detection results for frame_0.jpg.
Found 27 detection results for frame_10784.jpg.
Found 5 detection results for frame_13874.jpg.
Found 37 detection results for frame_13395.jpg.
Found 1 detection results for frame_14236.jpg.
Found 11 detection results for frame_14516.jpg.
Found 31 detection results for frame_15243.jpg.
Processing frame: frame_10807.jpg
Processing frame: frame_13412.jpg
Found 22 detection results for frame_11933.jpg.
Processing frame: frame_10065.jpg
Processing frame: frame_14265.jpgProcessing frame: frame_13919.jpg

Processing frame: frame_14542.jpg
Processing frame: frame_11934.jpg
Processing frame: frame_15309.jpg
Found 37 detection results for frame_13412.jpg.
Found 19 detection results for fram

In [ ]:
import json

# Kiểm tra file JSON
json_file = '/content/drive/MyDrive/L07_new/results/L07_V002_keyframes_filtered.json'

try:
    with open(json_file, 'r') as f:
        data = json.load(f)
        print("JSON file loaded successfully.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON: {e}")


JSON file loaded successfully.


In [ ]:
import os
import json
import time
import gc  # Thư viện để thu gom rác
from PIL import Image
from concurrent.futures import ProcessPoolExecutor
from parseq.strhub.data.module import SceneTextDataModule
from parseq.strhub.models.utils import load_from_checkpoint
import torch
from tqdm import tqdm

# Class để load và predict ảnh sử dụng PARSeq
class PARSeqPredictor:
    def __init__(self, checkpoint_path, device='cuda'):
        self.device = device
        self.parseq, self.img_transform = self.load_model_parseq(checkpoint_path, device)

    def load_model_parseq(self, checkpoint_path, device):
        parseq = load_from_checkpoint(checkpoint_path).eval().to(device)
        img_transform = SceneTextDataModule.get_transform(parseq.hparams.img_size)
        return parseq, img_transform

    @torch.inference_mode()
    def predict(self, image):
        image = self.img_transform(image).unsqueeze(0).to(self.device)
        p = self.parseq(image).softmax(-1)
        pred, p = self.parseq.tokenizer.decode(p)
        return pred, torch.mean(p[0])

# Hàm để crop ảnh dựa trên bounding box
def crop_image_in_memory(image, bd_pts):
    left = bd_pts[0][0]
    top = bd_pts[0][1]
    right = bd_pts[2][0]
    bottom = bd_pts[2][1]
    cropped_image = image.crop((left, top, right, bottom))
    return cropped_image

# Hàm xử lý một frame ảnh
def process_frame(frame_data):
    try:
        frame_id = frame_data['frame_id']
        detection_results = frame_data['detection_results']
        image_dir = frame_data['image_dir']
        checkpoint_path = frame_data['checkpoint_path']
        predictor = PARSeqPredictor(checkpoint_path)  # Khởi tạo tại đây

        image_path = os.path.join(image_dir, frame_id)
        if os.path.exists(image_path):
            image = Image.open(image_path)

            # Duyệt qua các bounding box và crop ảnh
            for idx, result in enumerate(detection_results):
                bd_pts = result["bd_pts"]
                cropped_image = crop_image_in_memory(image, bd_pts)
                pred_text, confidence = predictor.predict(cropped_image)
                print(f"Predicted text: {pred_text}, Confidence: {confidence:.4f}, type: {type(pred_text)}")
        else:
            print(f"Image {frame_id} not found in directory {image_dir}")
    except Exception as e:
        print(f"Error processing frame {frame_id}: {e}")
    finally:
        # Xóa cache GPU và thu gom rác RAM
        torch.cuda.empty_cache()  # Giải phóng bộ nhớ GPU
        gc.collect()  # Thu gom rác RAM

# Hàm xử lý ảnh từ file JSON
def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4):
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Chuẩn bị dữ liệu cho quá trình xử lý
    frame_data_list = [{'frame_id': frame_id,
                        'detection_results': data[frame_id]['detection_results'],
                        'image_dir': image_dir,
                        'checkpoint_path': checkpoint_path}
                       for frame_id in data.keys()]

    # Sử dụng ProcessPoolExecutor để xử lý
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        list(tqdm(executor.map(process_frame, frame_data_list), total=len(frame_data_list)))

# Thực thi chương trình
if __name__ == "__main__":
    json_file = '/content/drive/MyDrive/L07_new/results/L07_V001_keyframes_filtered.json'
    image_dir = '/content/drive/MyDrive/L07_new/L07_V001_keyframes_filtered'
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=1)
    end_time = time.time()

    print(f"Tổng thời gian xử lý: {end_time - start_time} giây")


  0%|          | 0/735 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import os
import json
import time
from concurrent.futures import ThreadPoolExecutor
%cd '/content/Run_parseq_ocr'

from main import predict


def process_single_image(image_path, checkpoint_path):
    """Process a single pre-cropped image."""
    try:
        predict(checkpoint_path, image_path)
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")

def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, max_frames=100):
    """Process pre-cropped images based on data from a JSON file."""
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Lấy max_frames frame đầu tiên từ file JSON
    frame_ids = list(data.keys())[:max_frames]

    image_paths = []
    for frame_id in frame_ids:
        frame_data = data[frame_id]
        for idx, result in enumerate(frame_data["detection_results"]):
            # Giả sử tên file ảnh đã crop có định dạng: {frame_id}_{idx}.jpg
            image_name = f"{frame_id}_{idx}.jpg"
            image_path = os.path.join(image_dir, image_name)
            if os.path.exists(image_path):
                image_paths.append(image_path)
            else:
                print(f"Image {image_name} not found in directory {image_dir}")

    # Sử dụng ThreadPoolExecutor để xử lý nhiều ảnh cùng lúc
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_image, image_path, checkpoint_path)
                   for image_path in image_paths]

    # Đợi tất cả các luồng hoàn tất
    for future in futures:
        future.result()

if __name__ == "__main__":
    json_file = '/content/drive/MyDrive/L07_new/results/L07_V001_keyframes_filtered.json'
    image_dir = '/content/drive/MyDrive/L07_new/L07_V001_keyframes_filtered'
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=2, max_frames=100)
    end_time = time.time()

    print(f"Tổng thời gian xử lý: {end_time - start_time} giây")

/content/Run_parseq_ocr


ImportError: cannot import name 'predict' from 'main' (/content/Run_parseq_ocr/main.py)

In [ ]:
import os
import json
import time
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO
from main import predict
import tempfile

def crop_image_in_memory(image, bd_pts):
    """Crop image in memory based on the bounding box."""
    left, top = bd_pts[0]
    right, bottom = bd_pts[2]
    return image.crop((left, top, right, bottom))

def process_single_image(frame_id, frame_data, image_dir, checkpoint_path):
    """Process a single image frame."""
    image_path = os.path.join(image_dir, frame_id)
    if not os.path.exists(image_path):
        print(f"Image {frame_id} not found in directory {image_dir}")
        return

    image = Image.open(image_path)
    detection_results = frame_data["detection_results"]

    for idx, result in enumerate(detection_results):
        bd_pts = result["bd_pts"]
        cropped_image = crop_image_in_memory(image, bd_pts)

        # Sử dụng NamedTemporaryFile để tạo file tạm thời
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as temp_file:
            cropped_image.save(temp_file, format='JPEG')
            temp_file_path = temp_file.name

        try:
            # Gọi hàm predict với file tạm thời
            predict(checkpoint_path, temp_file_path)
        finally:
            # Đảm bảo xóa file tạm sau khi sử dụng
            os.unlink(temp_file_path)

def process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=4, max_frames=100):
    """Process images by reading bounding box data from a JSON file, limiting to max_frames."""
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Lấy max_frames frame đầu tiên từ file JSON
    frame_ids = list(data.keys())[:max_frames]

    # Sử dụng ThreadPoolExecutor để xử lý nhiều ảnh cùng lúc
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_image, frame_id, data[frame_id], image_dir, checkpoint_path)
                   for frame_id in frame_ids]

    # Đợi tất cả các luồng hoàn tất
    for future in futures:
        future.result()

if __name__ == "__main__":
    json_file = '/content/drive/MyDrive/L07_new/results/L07_V001_keyframes_filtered.json'
    image_dir = '/content/drive/MyDrive/L07_new/L07_V001_keyframes_filtered'
    checkpoint_path = '/content/checkpoint/weights/rec/best-parseq.ckpt'

    start_time = time.time()
    process_images_from_json(json_file, image_dir, checkpoint_path, max_workers=2, max_frames=100)
    end_time = time.time()

    print(f"Tổng thời gian xử lý: {end_time - start_time} giây")